<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/line_webhook_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LINE Webhook Bot（Colab動作確認版）

 **Flask + ngrok** 署名検証と返信のロジックは元のコードと同じです。

**仕組み**
- Colab上でFlaskサーバーを起動 → ngrokで一時的に外部公開
- 表示される `https://xxxx.ngrok-free.app/callback` を LINE Developers の Webhook URL に設定

**注意（本番運用には不向き）**
- Colabランタイムは時間で切れる（アイドル90分・最大12時間程度）。切れるとサーバーも停止します。
- ngrok無料版は起動のたびにURLが変わるため、毎回Webhook URLの貼り直しが必要です。
- 常時稼働させたいなら、元のCGIをレンタルサーバー（ロリポップ等）に置く方が適切です。

セルは上から順に実行してください。

# 必要な準備


* Line 公式アカウントを作っておく。  
* LINE DevelopersでCHANNEL_SECRETとCHANNEL_ACCESS_TOKENを取得しておく
* 外部からColabへのアクセスを可能にする（一時的にしかできない）ために、NGROK_AUTHTOKEN を取得しておく



## 1. ライブラリのインストール

In [ ]:
!pip install flask pyngrok --quiet
print("done")

## 2. 設定値の入力

- `CHANNEL_SECRET` / `CHANNEL_ACCESS_TOKEN` … LINE Developers の値
- `NGROK_AUTHTOKEN` … https://dashboard.ngrok.com/get-started/your-authtoken で取得（無料登録）

In [ ]:
CHANNEL_SECRET = "ここにChannel secret"
CHANNEL_ACCESS_TOKEN = "ここにChannel access token"
NGROK_AUTHTOKEN = "ここにngrokのauthtoken"

LINE_REPLY_URL = "https://api.line.me/v2/bot/message/reply"

## 3. 署名検証と返信の関数

元のCGIコードと同じ処理です。違いは入力の取り方だけ（CGI: `sys.stdin` / 環境変数 → Flask: `request`）。

In [ ]:
import hmac
import hashlib
import base64
import json
import urllib.request


def verify_signature(body_bytes, signature):
    """LINE署名検証。Channel secret でHMAC-SHA256を計算し、
    送られてきた署名と一致するか確認する。生のバイト列を使うのが重要。"""
    if not CHANNEL_SECRET or CHANNEL_SECRET == "ここにChannel secret":
        # 未設定時は検証スキップ（動作確認用。本番では危険）
        return True
    digest = hmac.new(
        CHANNEL_SECRET.encode("utf-8"),
        body_bytes,
        hashlib.sha256,
    ).digest()
    expected = base64.b64encode(digest).decode("utf-8")
    return hmac.compare_digest(expected, signature)


def reply_to_line(reply_token, text):
    """replyTokenを使ってテキストを返信する。reply_tokenは短時間しか使えない。"""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {CHANNEL_ACCESS_TOKEN}",
    }
    data = {
        "replyToken": reply_token,
        "messages": [{"type": "text", "text": text}],
    }
    req = urllib.request.Request(
        LINE_REPLY_URL,
        data=json.dumps(data).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    urllib.request.urlopen(req)

print("関数を定義しました")

## 4. Flaskアプリの定義

CGI版のmain()` に相当する部分を Flask の `/callback` エンドポイントに移植してある。

| CGI版 | Flask版 |
|---|---|
| `sys.stdin.buffer.read()` | `request.get_data()` |
| `os.environ["HTTP_X_LINE_SIGNATURE"]` | `request.headers.get("X-Line-Signature")` |
| `print("Status: 200 OK")` | `return "OK", 200` |

In [ ]:
from flask import Flask, request

app = Flask(__name__)


@app.route("/callback", methods=["POST"])
def callback():
    # 1. 生のバイト列で本文を取得（署名検証に必要）
    body_bytes = request.get_data()

    # 2. ヘッダから署名を取得
    signature = request.headers.get("X-Line-Signature", "")

    # 3. 署名検証
    if not verify_signature(body_bytes, signature):
        return "NG", 400

    # 4. JSONとして解釈
    data = json.loads(body_bytes.decode("utf-8"))

    # 5. イベントを1件ずつ処理
    for event in data.get("events", []):
        if event.get("type") != "message":
            continue
        message = event.get("message", {})
        if message.get("type") != "text":
            continue

        # 6. 必要な3つの値
        reply_token = event.get("replyToken")
        user_id = event.get("source", {}).get("userId", "不明")
        user_text = message.get("text", "")

        # 7. エコーバックの返信文
        reply_text = f"ユーザーID: {user_id}\nメッセージ: {user_text}"

        # 8. 返信
        try:
            reply_to_line(reply_token, reply_text)
        except Exception as e:
            print("返信エラー:", e)

    return "OK", 200


@app.route("/", methods=["GET"])
def index():
    return "LINE bot is running."

print("Flaskアプリを定義しました")

## 5. ngrokでトンネルを開いてサーバー起動

実行すると `Webhook URL: https://xxxx.ngrok-free.app/callback` が表示されます。
これを LINE Developers のチャネル設定の **Webhook URL** に貼り付け、「検証」を押してください。

> このセルは起動したまま（実行中のまま）になります。停止するとサーバーも止まります。

In [ ]:
from pyngrok import ngrok, conf
import threading

# 既存トンネルを掃除
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

conf.get_default().auth_token = NGROK_AUTHTOKEN

PORT = 5000
public_url = ngrok.connect(PORT).public_url
print("=" * 50)
print(f"Webhook URL: {public_url}/callback")
print("↑ これを LINE Developers の Webhook URL に設定")
print("=" * 50)

# Flaskを別スレッドで起動（Colabのセルをブロックしないため）
def run():
    app.run(port=PORT, use_reloader=False)

threading.Thread(target=run, daemon=True).start()
print("サーバー起動中。LINEから送信してテストしてください。")

## 補足：動作確認の手順

1. LINE Developers → Messaging API設定 → Webhook URL に上記URLを設定し「検証」
2. 「Webhookの利用」をオンにする
3. 自動応答メッセージは必要に応じてオフ（任意）
4. ボットを友だち追加して、トークでメッセージを送信
5. 「ユーザーID: ... / メッセージ: ...」が返ってくれば成功

うまくいかないときは、上のセル4のログ（`返信エラー`）やngrokの画面を確認してください。